# 01. Ingest Structured CSV
Reads the raw AyurGenixAI CSV, cleans it, engineers the `text_chunk` feature, and saves it as a Delta Table.

In [ ]:
import re
from pyspark.sql.functions import col, concat_ws, lit

# --- Configuration ---
CATALOG = "main"
SCHEMA = "ayurgenix"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/files"
CSV_PATH = f"{VOLUME_PATH}/AyurGenixAI_Dataset.csv"
DELTA_TABLE = f"{CATALOG}.{SCHEMA}.ayurgenix_structured"

In [ ]:
print("📊 Loading and cleaning CSV...")

try:
    csv_df = spark.read \
        .option("header", "true") \
        .option("multiLine", "true") \
        .option("escape", '"') \
        .csv(CSV_PATH)

    # Clean column names
    def clean_column(name):
        name = name.strip()
        name = re.sub(r"[ &()/]", "_", name)
        name = re.sub(r"_+", "_", name)
        return name.lower()

    csv_df = csv_df.toDF(*[clean_column(c) for c in csv_df.columns])
    csv_df = csv_df.fillna("Not Available")

    # Ensure column exists or fallback gracefully
    def safe_col(df, c_name):
        if c_name in df.columns:
            return col(c_name)
        else:
            return lit("Not Available")

    # Create rich text chunk for RAG
    structured_df = csv_df.withColumn(
        "text_chunk",
        concat_ws(" | ",
            lit("Disease:"), safe_col(csv_df, "disease"),
            lit("Symptoms:"), safe_col(csv_df, "symptoms"),
            lit("Severity:"), safe_col(csv_df, "symptom_severity"),
            lit("Diagnosis:"), safe_col(csv_df, "diagnosis_tests"),
            lit("Doshas:"), safe_col(csv_df, "doshas"),
            lit("Herbs:"), safe_col(csv_df, "ayurvedic_herbs"),
            lit("Formulation:"), safe_col(csv_df, "formulation"),
            lit("Diet:"), safe_col(csv_df, "dietary_habits"),
            lit("Lifestyle:"), safe_col(csv_df, "diet_and_lifestyle_recommendations"),
            lit("Yoga:"), safe_col(csv_df, "yoga_physical_therapy"),
            lit("Complications:"), safe_col(csv_df, "complications"),
            lit("Recommendations:"), safe_col(csv_df, "patient_recommendations")
        )
    ).select(
        lit("CSV").alias("source"),
        safe_col(csv_df, "disease").alias("topic"),
        col("text_chunk")
    )

    print(f"💾 Saving {structured_df.count()} rows to {DELTA_TABLE}...")

    structured_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(DELTA_TABLE)

    display(spark.table(DELTA_TABLE).limit(5))

except Exception as e:
    print(f"⚠️ Error processing CSV: {e}")

**Next Step:** Run `02_ingest_unstructured_texts` to process the Ayurvedic PDFs.